<h1>Libraries</h1>

In [ ]:
import dask.dataframe as dd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

<h1>Part 1</h1>

<h3>Load dataset and categorize columns</h3>

In [ ]:
df = dd.read_csv("data.csv")

categorical_columns = [
    "Label",
    "Traffic Type",
    "Traffic Subtype",
    "Protocol",
]

# List of columns to exclude
unnecessary_columns = [
    "Flow ID", "Timestamp", "Src IP", "Dst IP", "Src Port", "Dst Port"
]

binary_columns = [

    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
]

# Filter out the unnecessary ones
not_numerical_columns = categorical_columns + unnecessary_columns + binary_columns
numerical_columns = [col for col in df.columns.tolist() if col not in not_numerical_columns]
num_df = df[numerical_columns]

<h3>Compute Basic Stats</h3>

In [ ]:
stats = num_df.describe().compute()

In [ ]:
correlation = num_df.corr().compute()

In [ ]:
stats_correlation = stats.T.corr()

<h3>Print Basic Stats</h3>

In [ ]:
print(stats)
stats.to_csv("csv/stats.csv")

<h3>Plot Basic Stats</h3>

In [ ]:
for idx, row in stats.iterrows():
    if idx == "count":
        continue
    plt.figure()
    row.plot(figsize=(20,10), kind='bar')

    plt.title(idx)
    plt.xlabel("Columns")
    plt.ylabel("Values")
    plt.tight_layout()
    plt.show()

<h2>Correlation between columns</h2>

<h3>Heatmap</h3>

In [ ]:
# Mask upper triangle
mask = np.triu(np.ones_like(correlation, dtype=bool))

plt.figure(figsize=(20, 18))  # Increase size
sns.heatmap(correlation, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, annot=False, cbar_kws={"shrink": 0.5})
plt.title('Correlation Heatmap (Masked Upper Triangle)')
plt.show()

<h3>Print highest correlations</h3>

In [ ]:
high_corr_pairs = correlation.where(mask).stack()  # This converts the DataFrame to a Series with MultiIndex
high_corr_pairs = high_corr_pairs[high_corr_pairs > 0.75]  # Filter the correlations greater than 0.75

# Print the results
print("Correlation pairs")
for idx, value in high_corr_pairs.sort_values(ascending=False).items():
    if idx[0] == idx[1]:
        # Skip self-correlation
        continue
    
    print(f"{value:.2f} {idx[0]} - {idx[1]}")

<h2>Statistical Correlation</h2>

<h3>Heatmap</h3>

In [ ]:
# Mask upper triangle
mask = np.triu(np.ones_like(stats_correlation, dtype=bool))

plt.figure(figsize=(20, 18))  # Increase size
sns.heatmap(stats_correlation, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, annot=False, cbar_kws={"shrink": 0.5})
plt.title('Correlation Heatmap (Masked Upper Triangle)')
plt.show()

<h3>Print</h3>

In [ ]:
print("Statistical Correlation pairs")
for idx, value in high_corr_pairs.sort_values(ascending=False).items():
    if idx[0] == idx[1]:
        # Skip self-correlation
        continue
    print(f"{value:.2f} {idx[0]} - {idx[1]}")

Fallback if correlation variable is lost

In [ ]:
correlation = dd.read_csv("csv/orrelation.csv").compute() 

# Get column names
col_names = correlation.columns.tolist()

# Create a new column with the column names as rows
correlation.insert(0, '', col_names)
correlation.set_index('', inplace=True)